# Data Profiling and Automated Report Generation
This notebook provides a tool to upload datasets, analyze them, and generate a professional Word document containing data profiling, analysis, and recommendations.

In [ ]:
# !pip install python-docx pandas openpyxl

In [ ]:
import os
import pandas as pd
from google.colab import files
from docx import Document
from docx.shared import Pt

# 1. Setup Output Folder
output_dir = 'Output'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 2. File Upload
print("Please upload your Excel (.xlsx) or CSV (.csv) file:")
uploaded = files.upload()
file_path = list(uploaded.keys())[0]

# Load data
if file_path.endswith('.csv'):
    df = pd.read_csv(file_path)
else:
    df = pd.read_excel(file_path)

# 3. Profiling and Report Generation
doc = Document()
doc.add_heading('Data Profiling & Analysis Report', 0)

# Column-wise Analysis
doc.add_heading('Column-Specific Analysis', level=1)

for col in df.columns:
    # Profiling data
    dtype = df[col].dtype
    null_count = df[col].isnull().sum()
    unique_count = df[col].nunique()
    
    # Dynamic Analysis Logic
    analysis = f"This column is of type {dtype}. It contains {null_count} missing values and {unique_count} unique entries."
    
    # Dynamic Recommendation Logic
    if null_count > 0:
        recommendation = "Consider imputing missing values or investigating the cause of data gaps."
    elif unique_count == 1:
        recommendation = "This column has constant values and may be dropped if it provides no predictive power."
    else:
        recommendation = "Data looks consistent for this attribute."

    # Add to Document as bullet points
    p = doc.add_paragraph(style='List Bullet')
    run = p.add_run(f"Column: {col}")
    run.bold = True
    
    doc.add_paragraph(f"• Data Profile: Type: {dtype}, Nulls: {null_count}, Uniques: {unique_count}", style='List Bullet 2')
    doc.add_paragraph(f"• Analysis Report: {analysis}", style='List Bullet 2')
    doc.add_paragraph(f"• Recommendation: {recommendation}", style='List Bullet 2')

# 4. Global Recommendation
doc.add_heading('General Recommendations', level=1)
summary_text = (
    "Based on the overall assessment, the dataset shows variations in data quality. "
    "It is recommended to standardize naming conventions, handle missing values identified in the specific column analysis, "
    "and ensure data types are optimized for downstream processing (e.g., converting objects to categories where appropriate). "
    "Further outlier detection is advised for numerical features."
)
doc.add_paragraph(summary_text)

# 5. Save
output_filename = os.path.join(output_dir, 'Analysis_Report.docx')
doc.save(output_filename)

print(f"Success! The report has been saved to: {output_filename}")